# Todo
- Explore the bid-ask spread across different maturities and strikes
- Explore the bid-ask spread across different underlying assets
- Explore the spread wrt to vega, implied volatility, etc.
- Explore the spread for a single day
- See the impact of transaction cost on backtest result
  - Try implementing another tcost model
- Backtest delta hedging strategies
- Backtest delta-gamma hedging strategies
  - Try other variant on calls or puts.

In [1]:
# Setup auto reload
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rc("font", **{"size": 18})
import numpy as np
from warnings import filterwarnings
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

filterwarnings("ignore")

import investment_lab.option_selection as option_selection
import investment_lab.backtest as backtest
from investment_lab.data.option_db import OptionLoader
from investment_lab.data.rates_db import USRatesLoader
from investment_lab.option_trade import OptionTrade, DeltaHedgedOptionTrade, DeltaGammaHedgedOptionTrade, VarianceSwap
from investment_lab.backtest import StrategyBacktester, BacktesterBidAskFromData, BacktesterFixedRelativeBidAsk

from investment_lab.pricing.black_scholes import black_scholes_price, black_scholes_greeks
from investment_lab.rates import compute_forward
from investment_lab import option_strategies as option_strategies

In [3]:
df = VarianceSwap.generate_trades(
    datetime(2020, 1, 2),
    datetime(2020, 4, 30),
    tickers="SPY",
    legs=[{
    "day_to_expiry_target": 30,
    "strike_spacing": 2,
    "weight": 1,
    "rebal_week_day": [2]}
          ],
    cost_neutral=False,
)
df

2026-02-07 23:44:07,486 | INFO | Loading option data from 2020-01-02 00:00:00 to 2020-04-30 00:00:00
2026-02-07 23:44:07,489 | INFO | Reading between 2020-01-02 00:00:00 2020-04-30 00:00:00 from ..//data/optiondb_2016_2023.parquet with None
2026-02-07 23:44:30,731 | INFO | Processing with {'ticker': 'SPY'}
2026-02-07 23:44:30,744 | INFO | Potentially add extra field with None
2026-02-07 23:44:51,697 | INFO | Preprocessing option data.
2026-02-07 23:44:51,995 | INFO | Selecting options for leg: VARIANCE SWAP using the rules:
{'day_to_expiry_target': 30}
2026-02-07 23:44:53,045 | INFO | Converting 2319 df_trades to daily time series
2026-02-07 23:44:56,700 | INFO | Forward filling option data for df
2026-02-07 23:45:01,849 | INFO | Reading between 2020-01-08 00:00:00 2020-04-30 00:00:00 from ..//data/par-yield-curve-rates-2020-2023.csv with None
2026-02-07 23:45:01,967 | INFO | Processing with None
2026-02-07 23:45:01,967 | INFO | Potentially add extra field with None
2026-02-07 23:45:04

,date,option_id,entry_date,leg_name,weight,ticker
0,2020-01-08,SPY,2020-01-08,DELTA_HEDGING,0.007099,SPY
1,2020-01-08,SPY 20200207C325,2020-01-08,VARIANCE SWAP,0.003083,SPY
2,2020-01-08,SPY 20200207C326,2020-01-08,VARIANCE SWAP,0.003083,SPY
3,2020-01-08,SPY 20200207C327,2020-01-08,VARIANCE SWAP,0.003083,SPY
4,2020-01-08,SPY 20200207C328,2020-01-08,VARIANCE SWAP,0.003083,SPY
...,...,...,...,...,...,...
42492,2020-04-30,SPY 20200529P289,2020-04-29,VARIANCE SWAP,0.003411,SPY
42493,2020-04-30,SPY 20200529P290,2020-04-29,VARIANCE SWAP,0.003411,SPY
42494,2020-04-30,SPY 20200529P291,2020-04-29,VARIANCE SWAP,0.003411,SPY
42495,2020-04-30,SPY 20200529P292,2020-04-29,VARIANCE SWAP,0.003411,SPY


In [4]:
df.groupby("date")['option_id'].count()

date
2020-01-08     63
2020-01-09     63
2020-01-10     63
2020-01-13     63
2020-01-14     63
             ... 
2020-04-24    737
2020-04-27    593
2020-04-28    593
2020-04-29    693
2020-04-30    693
Name: option_id, Length: 82, dtype: int64